# **Library**

In [1]:
import numpy as np
import pandas as pd
from importlib import reload
from pathlib import Path
from modules import preprocess

In [2]:
reload(preprocess)

# **Data**

In [3]:
current_dir_path, data_dir_path = preprocess.set_data_path('Data')
atm_df_raw = preprocess.read_file(data_dir_path, 'ATM_IV.xlsx', pd.read_excel, sheet_name = 'ATMF')
smile_df_raw = preprocess.read_file(data_dir_path, 'Smile_Dec_2026.csv', pd.read_csv, header = [0, 1], index_col = [0, 1], parse_dates = [0])
surface_df_raw = preprocess.read_file(data_dir_path, 'Surface.csv', pd.read_csv, header = [0, 1], index_col = [0, 1], parse_dates = [0])

In [5]:
atm_df_raw.head()

,expiry,T,fwd,F_over_S,bracket_k,atmf_iv,atmf_var,iv_at_spot
0,2026-10-13,0.002740,7800.72,1.000232,5,0.093022,0.008653,0.0928
1,2026-10-14,0.005479,7801.12,1.000283,5,0.108673,0.011810,0.1086
2,2026-10-15,0.008219,7801.69,1.000356,5,0.107826,0.011626,0.1079
3,2026-10-16,0.010959,7803.88,1.000637,5,0.103406,0.010693,0.1035
4,2026-10-19,0.019178,7805.18,1.000804,5,0.093182,0.008683,0.0935


In [6]:
smile_df_raw.head()

Call                                \
                               Ticker     Bid     Ask     Last Volm   
Expiry     Strike                                                     
2026-12-18 200      SPX 12/18/26 C200  7577.9  7595.3  7586.79    3   
           400      SPX 12/18/26 C400  7377.4  7395.5  7306.55    0   
           600      SPX 12/18/26 C600  7181.2  7198.6  7198.37    0   
           800      SPX 12/18/26 C800  6979.1  7000.3  7003.58    2   
           1000    SPX 12/18/26 C1000  6784.5  6800.9  6809.43    1   

                                  Put                        
                               Ticker  Bid   Ask  Last Volm  
Expiry     Strike                                            
2026-12-18 200      SPX 12/18/26 P200  0.0  0.05  0.05    0  
           400      SPX 12/18/26 P400  0.0  0.05  0.03    0  
           600      SPX 12/18/26 P600  0.0  0.05  0.05    0  
           800      SPX 12/18/26 P800  0.0  0.05  0.05    0  
           1000    SPX 12/18/26 P1000  0.0  0.05  0.05    0

## **Forward for Oct 18, 2026 and Mar 19, 2027**

In [4]:
T0 = pd.Timestamp('2026-10-12')
fwd_dec = atm_df_raw.loc[atm_df_raw['expiry'] == '2026-12-18']['fwd'].iloc[0]
fwd_mar = atm_df_raw.loc[atm_df_raw['expiry'] == '2027-03-19']['fwd'].iloc[0]
fwd = [fwd_dec, fwd_mar]

## **Clean zero bid and low quantity**

In [5]:
smile_df = preprocess.clean_smile(smile_df_raw,
                                  0.0,
                                  1,
                                  'Both')
preprocess.mid_price(smile_df)

# **Get discount factor**

In [6]:
discount_factor_smile = preprocess.get_df_both(smile_df, fwd)
discount_factor_smile

{Timestamp('2026-12-18 00:00:00'): (np.float64(0.9919403235485809),
  np.float64(0.9938546849154778),
  np.float64(0.9909879228801187)),
 Timestamp('2027-03-19 00:00:00'): (np.float64(0.9780957297735824),
  np.float64(0.9788332472896233),
  np.float64(0.9773369007709471))}

In [7]:
smile_call_df = preprocess.clean_smile(
    smile_df_raw, 0.0, 10, 'Call')

smile_put_df = preprocess.clean_smile(
    smile_df_raw, 0.0, 10, 'Put')

In [8]:
preprocess.year_fraction(T0, smile_call_df)
preprocess.mid_price(smile_call_df, False)
preprocess.year_fraction(T0, smile_put_df)
preprocess.mid_price(smile_put_df, False)

In [9]:
preprocess.get_all_iv(smile_call_df, fwd, discount_factor_smile, 'call')
preprocess.get_all_iv(smile_put_df, fwd, discount_factor_smile, 'put')

In [10]:
preprocess.discrptive_stat(smile_call_df, fwd_dec, 'iv_spread', '2026-12-18', 'call')

,All iv_spread,OTM iv_spread,ITM iv_spread
count,49.000000,42.000000,7.000000
mean,0.001130,0.000712,0.003639
std,0.002048,0.000515,0.004814
min,0.000388,0.000388,0.000455
25%,0.000476,0.000477,0.000492
50%,0.000531,0.000530,0.000605
75%,0.000762,0.000744,0.005335
max,0.012760,0.003528,0.012760


In [11]:
preprocess.discrptive_stat(smile_call_df, fwd_mar, 'iv_spread', '2027-03-19', 'call')

,All iv_spread,OTM iv_spread,ITM iv_spread
count,21.000000,18.000000,3.000000
mean,0.000630,0.000527,0.001242
std,0.000598,0.000393,0.001265
min,0.000335,0.000335,0.000452
25%,0.000401,0.000396,0.000512
50%,0.000443,0.000425,0.000572
75%,0.000499,0.000472,0.001636
max,0.002701,0.002084,0.002701


In [12]:
preprocess.discrptive_stat(smile_put_df, fwd_mar, 'iv_spread', '2026-12-18', 'put')

,All iv_spread,OTM iv_spread,ITM iv_spread
count,112.000000,109.000000,3.000000
mean,0.002663,0.002724,0.000468
std,0.009860,0.009989,0.000009
min,0.000355,0.000355,0.000461
25%,0.000458,0.000457,0.000463
50%,0.000562,0.000578,0.000464
75%,0.001053,0.001055,0.000471
max,0.087183,0.087183,0.000478


In [13]:
preprocess.discrptive_stat(smile_put_df, fwd_dec, 'iv_spread', '2027-03-19', 'put')

,All iv_spread,OTM iv_spread,ITM iv_spread
count,55.000000,47.000000,8.000000
mean,0.000697,0.000739,0.000447
std,0.000687,0.000735,0.000066
min,0.000364,0.000364,0.000393
25%,0.000408,0.000414,0.000394
50%,0.000485,0.000511,0.000423
75%,0.000729,0.000761,0.000484
max,0.004820,0.004820,0.000555


In [14]:
smile_put_df.loc['2026-12-18', ['Bid', 'Ask' , 'Volm', 'iv_spread', 'Mid_iv']][smile_put_df.loc['2026-12-18'].index < 0.7*fwd_dec]

,Bid,Ask,Volm,iv_spread,Mid_iv
Strike,,,,,
2000,0.05,0.20,14,0.087183,0.933766
2200,0.10,0.20,152,0.043282,0.881328
2600,0.15,0.30,131,0.041079,0.791854
3500,0.65,0.80,13,0.012084,0.645888
4000,1.20,1.35,50,0.006747,0.574776
4350,1.75,1.90,20,0.004639,0.527090
4500,2.05,2.25,75,0.005235,0.508052
4600,2.30,2.50,85,0.004683,0.495634
4650,2.45,2.65,15,0.004412,0.489804


In [15]:
smile_put_df.loc['2027-03-19', ['Bid', 'Ask' , 'Volm', 'iv_spread', 'Mid_iv']][smile_put_df.loc['2027-03-19'].index < 0.7*fwd_mar]

,Bid,Ask,Volm,iv_spread,Mid_iv
Strike,,,,,
2600,1.95,2.1,120,0.004820,0.631405
4000,7.30,7.6,102,0.002537,0.465781
4300,9.30,9.5,800,0.001328,0.434693
4400,10.00,10.3,300,0.001841,0.424656
4600,11.70,12.0,85,0.001571,0.405144
5350,20.80,21.1,60,0.000882,0.335599
5400,21.70,22.0,86,0.000847,0.331386
5425,22.10,22.4,33,0.000832,0.329106
5500,23.50,23.8,362,0.000784,0.322694


In [16]:
strike = smile_put_df.index.get_level_values('Strike')
expiry = smile_put_df.index.get_level_values('Expiry')

remove_mask = (((expiry == '2026-12-18') & (strike < 0.6 * fwd_dec)) |
               ((expiry == '2027-03-19') & (strike < 0.6 * fwd_mar)))

clean_smile_put_df = smile_put_df[~remove_mask]

In [17]:
preprocess.discrptive_stat(clean_smile_put_df, fwd_dec, 'iv_spread', '2026-12-18', 'put')

,All iv_spread,OTM iv_spread,ITM iv_spread
count,103.000000,90.000000,13.000000
mean,0.000864,0.000926,0.000432
std,0.000801,0.000839,0.000039
min,0.000355,0.000355,0.000376
25%,0.000455,0.000462,0.000380
50%,0.000529,0.000580,0.000451
75%,0.000866,0.000957,0.000453
max,0.004707,0.004707,0.000478


In [18]:
preprocess.discrptive_stat(clean_smile_put_df, fwd_dec, 'iv_spread', '2027-03-19', 'put')

,All iv_spread,OTM iv_spread,ITM iv_spread
count,50.000000,42.000000,8.000000
mean,0.000525,0.000540,0.000447
std,0.000152,0.000160,0.000066
min,0.000364,0.000364,0.000393
25%,0.000403,0.000412,0.000394
50%,0.000477,0.000485,0.000423
75%,0.000559,0.000609,0.000484
max,0.000882,0.000882,0.000555


In [19]:
iv_put_dec  = clean_smile_put_df.loc['2026-12-18', ['Mid_iv', 'Ask_iv', 'Bid_iv', 'iv_spread']]
iv_call_dec = smile_call_df.loc['2026-12-18', ['Mid_iv', 'Ask_iv', 'Bid_iv', 'iv_spread']]

# out-of-the-money side only
iv_put_dec  = iv_put_dec[iv_put_dec.index < fwd_dec]
iv_call_dec = iv_call_dec[iv_call_dec.index >= fwd_dec]

iv_put_dec['type'] = 'put'
iv_call_dec['type'] = 'call'

iv_df_dec = pd.concat([iv_put_dec, iv_call_dec], axis=0).sort_index()

In [20]:
iv_df_dec

,Mid_iv,Ask_iv,Bid_iv,iv_spread,type
Strike,,,,,
4800,0.471461,0.473311,0.469568,0.003742,put
4900,0.458680,0.460355,0.456969,0.003386,put
4975,0.449265,0.451584,0.446878,0.004707,put
5000,0.445567,0.447093,0.444012,0.003081,put
5100,0.433587,0.434960,0.432191,0.002768,put
...,...,...,...,...,...
8625,0.110718,0.111230,0.110197,0.001033,call
8650,0.111359,0.111923,0.110783,0.001139,call
8700,0.112912,0.113422,0.112392,0.001030,call


In [21]:
iv_put_mar  = clean_smile_put_df.loc['2027-03-19', ['Mid_iv', 'Ask_iv', 'Bid_iv', 'iv_spread']]
iv_call_mar = smile_call_df.loc['2027-03-19', ['Mid_iv', 'Ask_iv', 'Bid_iv', 'iv_spread']]

# out-of-the-money side only
iv_put_mar  = iv_put_mar[iv_put_mar.index < fwd_mar]
iv_call_mar = iv_call_mar[iv_call_mar.index >= fwd_mar]

iv_put_mar['type'] = 'put'
iv_call_mar['type'] = 'call'

iv_df_mar = pd.concat([iv_put_mar, iv_call_mar], axis=0).sort_index()

iv_df_mar

,Mid_iv,Ask_iv,Bid_iv,iv_spread,type
Strike,,,,,
5350,0.335599,0.336039,0.335157,0.000882,put
5400,0.331386,0.331808,0.330961,0.000847,put
5425,0.329106,0.329521,0.328689,0.000832,put
5500,0.322694,0.323085,0.322301,0.000784,put
5550,0.318228,0.318605,0.317850,0.000755,put
...,...,...,...,...,...
8600,0.121058,0.121296,0.120819,0.000477,call
8650,0.120293,0.120508,0.120077,0.000432,call
8800,0.118454,0.118740,0.118168,0.000572,call


In [22]:
iv_df_mar.describe()

,Mid_iv,Ask_iv,Bid_iv,iv_spread
count,62.000000,62.000000,62.000000,62.000000
mean,0.195378,0.195643,0.195112,0.000531
std,0.064339,0.064386,0.064292,0.000247
min,0.117540,0.117824,0.117256,0.000335
25%,0.138201,0.138404,0.137997,0.000401
50%,0.182298,0.182488,0.182106,0.000450
75%,0.240483,0.240772,0.240193,0.000559
max,0.335599,0.336039,0.335157,0.002084


In [23]:
iv_df_dec.to_csv(data_dir_path / 'IV_Smile_Dec_2026.csv')
iv_df_mar.to_csv(data_dir_path / 'IV_Smile_Mar_2027.csv')